In [1]:
import pandas as pd

# Import caboost, lightgbm, xgboost
from catboost import CatBoostClassifier
import lightgbm as lgbm
import xgboost as xgb

# Import sklearn metrics
from sklearn.metrics import (accuracy_score,
                             f1_score,
                             average_precision_score,
                             roc_auc_score,
                             confusion_matrix,
                             classification_report)
from sklearn.model_selection import train_test_split

# Import VotingClassifier and StackingClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier

# Import LabelEncoder
from sklearn.preprocessing import LabelEncoder

# Import cross-validation
from sklearn.model_selection import KFold

# Init optuna to optimize weighting for VotingClassifier
import optuna
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Libraries for experiment tracker
import wandb
from optuna.integration.wandb import WeightsAndBiasesCallback

/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/NĂM 4/HKI/Intelligent Data Analysis/Chatbot_Extension/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv("data/preprocessed/train_cleaned.csv")
test_df = pd.read_csv("data/preprocessed/test_cleaned.csv")

In [3]:
train_df.head()

,id,Name,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression,SleepDuration_num,is_student
0,0,Aaradhya,Female,49.0,Ludhiana,Chef,0.0,5.0,0.00,0.0,2.0,Healthy,BHM,No,1.0,2.0,No,0,7.5,0
1,1,Vivan,Male,26.0,Varanasi,Teacher,0.0,4.0,0.00,0.0,3.0,Unhealthy,LLB,Yes,7.0,3.0,No,1,4.5,0
2,2,Yuvraj,Male,33.0,Visakhapatnam,Teacher,5.0,0.0,8.97,2.0,0.0,Healthy,B.Pharm,Yes,3.0,1.0,No,1,5.5,1
3,3,Yuvraj,Male,22.0,Mumbai,Teacher,0.0,5.0,0.00,0.0,1.0,Moderate,BBA,Yes,10.0,1.0,Yes,1,4.5,0
4,4,Rhea,Female,30.0,Kanpur,Business Analyst,0.0,1.0,0.00,0.0,1.0,Unhealthy,BBA,Yes,9.0,4.0,Yes,0,5.5,0


In [4]:
test_df.head()

,id,Name,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,SleepDuration_num,is_student
0,140700,Shivam,Male,53.0,Visakhapatnam,Judge,0.0,2.0,0.00,0.0,5.0,Moderate,LLB,No,9.0,3.0,Yes,4.5,0
1,140701,Sanya,Female,58.0,Kolkata,Educational Consultant,0.0,2.0,0.00,0.0,4.0,Moderate,B.Ed,No,6.0,4.0,No,4.5,0
2,140702,Yash,Male,53.0,Jaipur,Teacher,0.0,4.0,0.00,0.0,1.0,Moderate,B.Arch,Yes,12.0,4.0,No,7.5,0
3,140703,Nalini,Female,23.0,Rajkot,Teacher,5.0,0.0,6.84,1.0,0.0,Moderate,BSc,Yes,10.0,4.0,No,7.5,1
4,140704,Shaurya,Male,47.0,Kalyan,Teacher,0.0,5.0,0.00,0.0,5.0,Moderate,BCA,Yes,3.0,4.0,No,7.5,0


In [5]:
# Prepare data for training
drop_cols = ["id", "Name"]
X = train_df.drop(columns=drop_cols + ["Depression"])
y = train_df["Depression"]
X_test_submission = test_df.drop(
    columns=drop_cols
)

In [6]:
# Identify categorical columns (object type)
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
cat_cols

['Gender',
 'City',
 'Profession',
 'Dietary Habits',
 'Degree',
 'Have you ever had suicidal thoughts ?',
 'Family History of Mental Illness']

In [7]:
# Encode categorical variables
combined = pd.concat([X, X_test_submission], axis=0)

for col in cat_cols:
    le = LabelEncoder()
    # Convert to string to handle potential mixed types
    combined[col] = le.fit_transform(combined[col].astype(str))

combined.head()

,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,SleepDuration_num,is_student
0,0,49.0,62,13,0.0,5.0,0.00,0.0,2.0,11,50,0,1.0,2.0,0,7.5,0
1,1,26.0,118,71,0.0,4.0,0.00,0.0,3.0,32,92,1,7.0,3.0,0,4.5,0
2,1,33.0,123,71,5.0,0.0,8.97,2.0,0.0,11,34,1,3.0,1.0,0,5.5,1
3,1,22.0,79,71,0.0,5.0,0.00,0.0,1.0,22,44,1,10.0,1.0,1,4.5,0
4,0,30.0,46,12,0.0,1.0,0.00,0.0,1.0,32,44,1,9.0,4.0,1,5.5,0


In [8]:
# Split back into train and test
X = combined.iloc[: len(X)]
X_test_submission = combined.iloc[len(X) :]

In [9]:
best_catboost_params = {
    "iterations": 2000,
    "learning_rate": 0.01,
    "depth": 7,
    "threshold": 0.77312897607623,
}

best_xgboost_params = {
    "colsample_bytree": 0.7527796619608157,
    "n_estimators": 785,
    "learning_rate": 0.06506454595661652,
    "reg_lambda": 0.3249164498962406,
    "reg_alpha": 2.52176887536818,
    "max_depth": 10,
    "num_leaves": 256,
    "gamma": 0.030639620918430623,
    "threshold": 0.5265995390797443,
}

best_lightgbm_params = {
    "num_leaves": 240,
    "max_depth": 4,
    "learning_rate": 0.09995896705073841,
    "min_child_samples": 55,
    "subsample": 0.6511856126273549,
    "colsample_bytree": 0.5360644669787153,
    "lambda_l1": 0.00014623461031687177,
    "lambda_l2": 5.773602684816733,
    "threshold": 0.5107309048999379,
}

In [10]:
wandb_kwargs = {
    "entity": "team-csc17001-ida",
    "project": "depression-detection",
    "name": "optuna_ensemble_weight_ensemble_1",
}

wandb_callback = WeightsAndBiasesCallback(
    metric_name="valid_ap",     # We focus in Average Precision because this is a clinical task.
    wandb_kwargs=wandb_kwargs,  # passed to wandb.init(...)
    as_multirun=False,          # one W&B run for entire study
)

wandb: Currently logged in as: themetasetter (themetasetter-i-h-c-qu-c-gia-tp-hcm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [11]:
# Define objective function for Optuna
@wandb_callback.track_in_wandb()
def objective(trial):
    # Suggest weights for each model
    catboost_weight = trial.suggest_float("catboost_weight", 0.0, 1.0)
    xgboost_weight = trial.suggest_float("xgboost_weight", 0.0, 1.0)
    lightgbm_weight = trial.suggest_float("lightgbm_weight", 0.0, 1.0)
    threshold = trial.suggest_float("threshold", 0.0, 6.8)

    # Normalize weights
    total_weight = catboost_weight + xgboost_weight + lightgbm_weight
    catboost_weight /= total_weight
    xgboost_weight /= total_weight
    lightgbm_weight /= total_weight

    # Since threshold is not a valid params but we need it for later, we remove it from the params
    catboost_params = best_catboost_params.copy()
    catboost_params.pop("threshold", None)
    xgboost_params = best_xgboost_params.copy()
    xgboost_params.pop("threshold", None)
    lightgbm_params = best_lightgbm_params.copy()
    lightgbm_params.pop("threshold", None)

    # Initialize models with best parameters
    catboost_clf = CatBoostClassifier(**catboost_params, verbose=0)
    xgboost_clf = xgb.XGBClassifier(**xgboost_params, use_label_encoder=False, verbosity=0)
    lightgbm_clf = lgbm.LGBMClassifier(**lightgbm_params, verbose=-1)

    # Create VotingClassifier with suggested weights
    voting_clf = VotingClassifier(
        estimators=[
            ("catboost", catboost_clf),
            ("xgboost", xgboost_clf),
            ("lightgbm", lightgbm_clf),
        ],
        voting="soft",
        weights=[catboost_weight, xgboost_weight, lightgbm_weight],
    )

    # Cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    pr_auc_scores = []
    f1_scores = []
    accuracies = []
    roc_auc_scores = []
    for train_index, val_index in kf.split(X):
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]

        # Fit the model
        voting_clf.fit(X_train, y_train)

        # Predict probabilities
        y_probs = voting_clf.predict_proba(X_val)[:, 1]
        y_pred = (y_probs >= threshold).astype(int)

        # Calculate metrics
        pr_auc = average_precision_score(y_val, y_probs)
        f1 = f1_score(y_val, y_pred)
        accuracy = accuracy_score(y_val, y_pred)
        roc_auc = roc_auc_score(y_val, y_probs)

        pr_auc_scores.append(pr_auc)
        f1_scores.append(f1)
        accuracies.append(accuracy)
        roc_auc_scores.append(roc_auc)
        
    mean_pr_auc = np.mean(pr_auc_scores)
    mean_f1_score = np.mean(f1_scores)
    mean_accuracy = np.mean(accuracies)
    mean_roc_auc = np.mean(roc_auc_scores)

    # Log metrics to wandb
    wandb.log({
        "mean_pr_auc": mean_pr_auc,
        "mean_f1_score": mean_f1_score,
        "mean_accuracy": mean_accuracy,
        "mean_roc_auc": mean_roc_auc,
        "catboost_weight": catboost_weight,
        "xgboost_weight": xgboost_weight,
        "lightgbm_weight": lightgbm_weight,
        "threshold": threshold,
    })

    return mean_pr_auc

In [12]:
study = optuna.create_study(direction="maximize")

[I 2025-11-30 18:48:47,120] A new study created in memory with name: no-name-a2cc0d6d-3b12-47ed-bfcb-636d8355e905


In [13]:
study.optimize(
    objective,
    n_trials=200,
    callbacks=[wandb_callback],
    n_jobs=1,
)
wandb.finish()

[I 2025-11-30 18:50:48,351] Trial 0 finished with value: 0.9068740988661352 and parameters: {'catboost_weight': 0.8186759757860805, 'xgboost_weight': 0.6269451850799773, 'lightgbm_weight': 0.0015718409984899484, 'threshold': 5.3540512986617435}. Best is trial 0 with value: 0.9068740988661352.
wandb: WARNING Tried to log to step 0 that is less than the current step 1. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
[I 2025-11-30 18:52:42,402] Trial 1 finished with value: 0.9079697984592411 and parameters: {'catboost_weight': 0.6199427296967472, 'xgboost_weight': 0.0028452059934637175, 'lightgbm_weight': 0.964713552897071, 'threshold': 0.8861963343196751}. Best is trial 1 with value: 0.9079697984592411.
wandb: WARNING Tried to log to step 1 that is less than the current step 2. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.

catboost_weight,▃▂▅▅▇█▁▇█▅▄▆▅▄▇▇▇▁▂▇▇▇▇▆▇▆▆▇▇▇▇▇▇▇▇▇█▇▄▇
lightgbm_weight,█▅█▂▄▁▅▃▇▆▅▃▄▅▁▆▆▅▅▅▅▅▆▅▆▄▄▃▅▅▄▄▄▃▆▅▅▅▅▅
mean_accuracy,▆▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
mean_f1_score,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
mean_pr_auc,▁▇█▇████▇▇▇█████▇████████████████▇██████
mean_roc_auc,▅▁▅▇██▇██▅▄███▇▇█▆███▇████████████▇█▇███
threshold,▇▂▄▂▄▆▇▅▅▁▆▄▅▄▄▅▄▂▅▄▄▄▄▃▄▆▇▆▆▇▇▅▅█▅▇▇▆▇▆
xgboost_weight,▄▁▂▅▃▂▂▆▂▂██▂▂▃▃█▁▂▃▂▃▂▂▃▁▂▁▂▂▁▂▂▂▁▂▄▅▂▁
catboost_weight,0.75118
lightgbm_weight,0.21875
mean_accuracy,0.81829


In [14]:
# Get the best hyper params from study object
best_params = study.best_trial.params
best_params

{'catboost_weight': 0.6331398224099682,
 'xgboost_weight': 0.04859523394160956,
 'lightgbm_weight': 0.18648058631740952,
 'threshold': 4.035494941443986}

In [15]:
print(best_params)

{'catboost_weight': 0.6331398224099682, 'xgboost_weight': 0.04859523394160956, 'lightgbm_weight': 0.18648058631740952, 'threshold': 4.035494941443986}
